In [1]:
import random

import boto3
import numpy as np
import pytz

from Data_query.trino_config import *

session = boto3.Session()
s3 = boto3.client("s3")

In [13]:
stop_trino()

Trino service stopping triggered.


In [11]:
big_workers = 1
workers = 0
num_workers = max(workers, big_workers)
ensure_trino_running(worker_desired_count=workers, big_worker_desired_count=big_workers)
sleep(60)  # wait for trino to be ready

Trino service is not running. Starting the service...
Trino service triggered.
Service trino-service is now stable.


In [ ]:
v1 = 253
v2 = 260


def run_func(args):
    year, month, split_cons = args
    df = iceberg_sql(f"""
                with data as (
                    select site_id, t_stamp, sum(power*circuit_polarity/1000) as P_kW, 
                            avg(voltage) as V, max(ac_capacity_kw) as ac_capacity_kw
                    from 
                    (select circuit_id, t_stamp, power, voltage 
                        from ts where year={year} and month={month} and is_pv=True and voltage > 0 and voltage < 300 and {split_cons}) as ts1
                inner join (select distinct site_id, circuit_id, circuit_polarity, ac_capacity_kw from meta_up23c where is_pv = True) as m on ts1.circuit_id = m.circuit_id
                group by site_id, t_stamp
                having avg(voltage) > 253),
                pq as (
                    select d.site_id, d.t_stamp, P_kW, d.V, ac_capacity_kw, vv.uncurtailed_P, vv.V as V_ghi,
                    (case when d.V < {v1} then ac_capacity_kw 
                    when d.V > {v2} then .2 * ac_capacity_kw
                    else ((ac_capacity_kw - .2 * ac_capacity_kw) / ({v1} - {v2}) * (d.V - {v2}) + .2 * ac_capacity_kw) end) + 0.04*ac_capacity_kw as max_P_volt_watt
                from data d left join voltwatt_uncurtailedPV as vv on d.site_id = vv.site_id and d.t_stamp = vv.t_stamp
                ),
                pq2 as (select site_id, t_stamp, day(t_stamp) as day, month(t_stamp) as month, year(t_stamp) as year, P_kW, uncurtailed_P, V, V_ghi, ac_capacity_kw, max_P_volt_watt,
                greatest(0, P_kW - max_P_volt_watt) as nonconformance_voltwattghi, 
                case when uncurtailed_P > max_P_volt_watt and P_kW < max_P_volt_watt then uncurtailed_P - P_kW else 0 end as curtailment_voltwattghi
                from pq 
                where uncurtailed_P > max_P_volt_watt or uncurtailed_P is null
                )
                
                select year, month, day, site_id, 
                sum(nonconformance_voltwattghi) as nonconformance_voltwattghi_sum,
                sum(curtailment_voltwattghi) as curtailment_voltwattghi_sum,
                sum(case when nonconformance_voltwattghi > 0 then 1 else 0 end) as nonconformance_voltwattghi_count,
                sum(case when curtailment_voltwattghi > 0 then 1 else 0 end) as curtailment_voltwattghi_count,
                count(nonconformance_voltwattghi) as total_count
                from pq2
                group by year, month, day, site_id
                order by nonconformance_voltwattghi_sum desc
                """)
    # sleep(random.randint(1, 10))  # add some randomness to avoid overwhelming trino with simultaneous queries
    print(
        f"Completed year={year}, month={month},  {split_cons.replace('system.bucket(postcode, 16)', 'bucket')}"
    )
    return df


tasks = [
    (year, month, split_cons)
    for year in (2024,)
    for month in range(1, 13)
    for split_cons in [
        "system.bucket(postcode, 16) <= 7",
        "system.bucket(postcode, 16) > 7",
    ]
]
df = trino_parallel(run_func, tasks, num_workers=num_workers)
df

Completed year=2024, month=1,  bucket <= 7
Completed year=2024, month=1,  bucket > 7
Completed year=2024, month=2,  bucket <= 7
Completed year=2024, month=2,  bucket > 7
Completed year=2024, month=3,  bucket <= 7
Completed year=2024, month=3,  bucket > 7
Completed year=2024, month=4,  bucket <= 7
Completed year=2024, month=4,  bucket > 7
Completed year=2024, month=5,  bucket <= 7
Completed year=2024, month=5,  bucket > 7
Completed year=2024, month=6,  bucket <= 7
Completed year=2024, month=6,  bucket > 7
Completed year=2024, month=7,  bucket <= 7
Completed year=2024, month=7,  bucket > 7
Completed year=2024, month=8,  bucket <= 7
Completed year=2024, month=8,  bucket > 7
Completed year=2024, month=9,  bucket <= 7
Completed year=2024, month=9,  bucket > 7
Completed year=2024, month=10,  bucket <= 7
Completed year=2024, month=10,  bucket > 7
Completed year=2024, month=11,  bucket <= 7
Completed year=2024, month=11,  bucket > 7
Completed year=2024, month=12,  bucket <= 7
Completed year=20

,year,month,day,site_id,nonconformance_voltwattghi_sum,curtailment_voltwattghi_sum,nonconformance_voltwattghi_count,curtailment_voltwattghi_count,null_uncurtailed_P_count,total_count
0,2024,1,22,755163749,394.168985,5.057239,68,4,5,77
1,2024,1,11,755163749,352.225633,0.000000,71,0,14,85
2,2024,1,31,755163749,314.239765,0.000000,56,0,3,59
3,2024,1,29,755163749,265.395384,0.000000,64,0,6,70
4,2024,1,23,755163749,256.659294,9.976520,52,4,4,60
...,...,...,...,...,...,...,...,...,...,...
197102,2024,12,10,1259467593,0.000000,0.000000,0,0,1,1
197103,2024,12,11,1259467593,0.000000,0.000000,0,0,5,5
197104,2024,12,12,1259467593,0.000000,9.510993,0,7,3,10
197105,2024,12,15,1259467593,0.000000,3.398370,0,3,3,6


In [ ]:
# 13 timestamps with conformant: P < threshold and 6 uncurtailed > threshold, 7

In [10]:
71 - 58

13

In [3]:
iceberg_sql("""
select * from all_uncurtailedPV limit 10
""")

,site_id,t_stamp,year,month,uncurtailed_p,ghi
0,3632088,2024-08-01 21:55:00,2024,8,0.784358,102.21
1,3632088,2024-08-01 22:00:00,2024,8,0.988446,132.11
2,3632088,2024-08-01 23:00:00,2024,8,2.259035,311.12
3,3632088,2024-08-01 23:10:00,2024,8,2.238457,263.11
4,3632088,2024-08-02 00:35:00,2024,8,3.293631,518.10
5,3632088,2024-08-02 04:25:00,2024,8,2.245790,380.67
6,3632088,2024-08-02 22:20:00,2024,8,1.325372,66.81
7,3632088,2024-08-02 22:25:00,2024,8,1.389546,66.81
8,3632088,2024-08-02 22:30:00,2024,8,1.266613,75.21
9,3632088,2024-08-02 22:35:00,2024,8,1.342566,75.21
